# MCP Gateway Test Client

Use this notebook to connect to the MCP gateway running in Docker, inspect its tools/resources, and call a tool.

Before running the cells:

1. Start the Docker gateway: `docker compose up -d --build gateway`
2. Confirm it is healthy: `docker compose ps gateway`
3. Select the project's `.venv` Python kernel in Cursor/Jupyter.

The notebook only connects to `MCP_URL`; it never starts or stops the gateway.

In [4]:
from typing import Any

from fastmcp import Client
from fastmcp.client.client import CallToolResult

MCP_URL = "http://localhost:8000/mcp/postgres"

In [5]:
class McpTestClient:
    """Inspect and call an MCP server through an existing endpoint."""

    def __init__(self, url: str) -> None:
        self._url = url

    async def inspect(self) -> None:
        print(f"Connecting to {self._url}")
        async with Client(self._url) as client:
            tools = await client.list_tools()
            print(f"\nTools ({len(tools)}):")
            for tool in tools:
                description = f" — {tool.description}" if tool.description else ""
                print(f"  {tool.name}{description}")

            resources = await client.list_resources()
            print(f"\nResources ({len(resources)}):")
            for resource in resources:
                print(f"  {resource.uri}")

    async def call_tool(
        self,
        tool_name: str,
        arguments: dict[str, Any],
    ) -> CallToolResult:
        async with Client(self._url) as client:
            result = await client.call_tool(tool_name, arguments)

        print(f"Result from {tool_name} (is_error={result.is_error}):")
        for content in result.content:
            text = getattr(content, "text", None)
            print(text if text is not None else content.model_dump_json(indent=2))
        return result

In [8]:
# Connects only to the MCP endpoint exposed by the Docker gateway.
mcp = McpTestClient(MCP_URL)
await mcp.inspect()

Connecting to http://localhost:8000/mcp/postgres

Tools (1):
  query — Run a read-only SQL query

Resources (1):
  postgres://aisafe@localhost:5432/customers/schema


## Call a tool

The example below runs a read-only query against the Postgres MCP. Change `tool_name` and `arguments` to test another MCP server.

In [9]:
result: CallToolResult = await mcp.call_tool(
    tool_name="query",
    arguments={
        "sql": "SELECT COUNT(*) AS customer_count FROM customers",
    },
)

Result from query (is_error=False):
[
  {
    "customer_count": "12"
  }
]


## Docker logs

The notebook does not manage the gateway process. View its requests with:

`docker compose logs -f gateway`